In [1]:
import os

print("--- 🔍 Scanning Data Directory to Find True Paths ---")
# Loop through the input directory to see exactly how Kaggle named the folder
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

--- 🔍 Scanning Data Directory to Find True Paths ---
/kaggle/input/competitions/open-problems-single-cell-perturbations/multiome_train.parquet
/kaggle/input/competitions/open-problems-single-cell-perturbations/multiome_obs_meta.csv
/kaggle/input/competitions/open-problems-single-cell-perturbations/sample_submission.csv
/kaggle/input/competitions/open-problems-single-cell-perturbations/adata_train.parquet
/kaggle/input/competitions/open-problems-single-cell-perturbations/multiome_var_meta.csv
/kaggle/input/competitions/open-problems-single-cell-perturbations/adata_obs_meta.csv
/kaggle/input/competitions/open-problems-single-cell-perturbations/id_map.csv
/kaggle/input/competitions/open-problems-single-cell-perturbations/de_train.parquet
/kaggle/input/competitions/open-problems-single-cell-perturbations/adata_excluded_ids.csv


In [2]:
import pandas as pd
import numpy as np

# 1. Load the core Differential Expression dataset using the correct path
print("--- 🔬 Initializing Digital Lab: Loading Data ---")
de_train = pd.read_parquet('/kaggle/input/competitions/open-problems-single-cell-perturbations/de_train.parquet')

print(f"✅ Success! Train Dataset Shape: {de_train.shape}")

# 2. Inspect the first 5 rows and the initial metadata + gene columns
print("\n🔍 Exploring the first 5 rows (Metadata + First 3 Genes):")
print(de_train.iloc[:5, :6])

--- 🔬 Initializing Digital Lab: Loading Data ---
✅ Success! Train Dataset Shape: (614, 18216)

🔍 Exploring the first 5 rows (Metadata + First 3 Genes):
            cell_type             sm_name sm_lincs_id  \
0            NK cells        Clotrimazole    LSM-5341   
1        T cells CD4+        Clotrimazole    LSM-5341   
2        T cells CD8+        Clotrimazole    LSM-5341   
3  T regulatory cells        Clotrimazole    LSM-5341   
4            NK cells  Mometasone Furoate    LSM-3349   

                                              SMILES  control      A1BG  
0             Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1    False  0.104720  
1             Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1    False  0.915953  
2             Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1    False -0.387721  
3             Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1    False  0.232893  
4  C[C@@H]1C[C@H]2[C@@H]3CCC4=CC(=O)C=C[C@]4(C)[C...    False  4.290652  


In [3]:
# Separate the metadata (Features) from the gene expression values (Targets)
# The first 5 columns are metadata, and the rest are the 18,211 genes
metadata_cols = ['cell_type', 'sm_name', 'sm_lincs_id', 'SMILES', 'control']

X_metadata = de_train[metadata_cols]
Y_genes = de_train.drop(columns=metadata_cols)

print("--- ⚙️ Feature vs Target Separation ---")
print(f"X Metadata Shape (Features): {X_metadata.shape}")
print(f"Y Genes Shape (Targets to predict): {Y_genes.shape}")

# Check the distribution of cell types in our training data
print("\n📊 Distribution of Immunological Cell Types:")
print(X_metadata['cell_type'].value_counts())

--- ⚙️ Feature vs Target Separation ---
X Metadata Shape (Features): (614, 5)
Y Genes Shape (Targets to predict): (614, 18211)

📊 Distribution of Immunological Cell Types:
cell_type
NK cells              146
T cells CD4+          146
T regulatory cells    146
T cells CD8+          142
B cells                17
Myeloid cells          17
Name: count, dtype: int64


In [4]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold

# 1. One-Hot Encode the categorical features (cell_type and sm_name)
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_encoded = encoder.fit_transform(de_train[['cell_type', 'sm_name']])

# Convert to DataFrame just to keep it clean and traceable
X_encoded_df = pd.DataFrame(X_encoded, columns=encoder.get_feature_names_out(['cell_type', 'sm_name']))

print("--- ⚙️ Feature Encoding Complete ---")
print(f"Encoded X Features Shape: {X_encoded_df.shape}")

# 2. Define the Competition Metric: Mean Rowwise Root Mean Squared Error (MRMSE)
def mean_rowwise_rmse(y_true, y_pred):
    # Calculate RMSE for each row independently, then compute the overall mean
    rowwise_rmse = np.sqrt(np.mean((y_true - y_pred) ** 2, axis=1))
    return np.mean(rowwise_rmse)

# 3. Cross-Validation Setup (3-Fold CV to validate our model's performance)
kf = KFold(n_splits=3, shuffle=True, random_state=42)
mrmse_scores = []

print("\n🚀 Training Multi-Output Ridge Regression Baseline...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X_encoded_df)):
    X_train, X_val = X_encoded_df.iloc[train_idx], X_encoded_df.iloc[val_idx]
    y_train, y_val = Y_genes.iloc[train_idx], Y_genes.iloc[val_idx]
    
    # Ridge regression handles 18,211 targets simultaneously with L2 regularization
    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train)
    
    # Predict and evaluate using MRMSE
    preds = model.predict(X_val)
    fold_score = mean_rowwise_rmse(y_val.values, preds)
    mrmse_scores.append(fold_score)
    print(f"Fold {fold + 1} MRMSE Score: {fold_score:.5f}")

print(f"\n🏆 Overall Baseline CV MRMSE: {np.mean(mrmse_scores):.5f}")

--- ⚙️ Feature Encoding Complete ---
Encoded X Features Shape: (614, 152)

🚀 Training Multi-Output Ridge Regression Baseline...
Fold 1 MRMSE Score: 1.31066
Fold 2 MRMSE Score: 1.42117
Fold 3 MRMSE Score: 1.18549

🏆 Overall Baseline CV MRMSE: 1.30577


In [5]:
from sklearn.neural_network import MLPRegressor

mrmse_nn_scores = []

print("🚀 Training Non-Linear Neural Network (MLP Regressor) Baseline...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_encoded_df)):
    X_train, X_val = X_encoded_df.iloc[train_idx], X_encoded_df.iloc[val_idx]
    y_train, y_val = Y_genes.iloc[train_idx], Y_genes.iloc[val_idx]
    
    # Using a 2-layer Multi-Layer Perceptron to capture non-linear drug-cell features
    # early_stopping=True ensures we don't overfit and saves computation time
    nn_model = MLPRegressor(
        hidden_layer_sizes=(128, 64), 
        activation='relu', 
        solver='adam', 
        max_iter=30, 
        early_stopping=True,
        random_state=42
    )
    
    nn_model.fit(X_train, y_train)
    
    # Predict and evaluate
    nn_preds = nn_model.predict(X_val)
    fold_score = mean_rowwise_rmse(y_val.values, nn_preds)
    mrmse_nn_scores.append(fold_score)
    print(f"Neural Net Fold {fold + 1} MRMSE Score: {fold_score:.5f}")

print(f"\n🏆 Overall Neural Network CV MRMSE: {np.mean(mrmse_nn_scores):.5f}")

🚀 Training Non-Linear Neural Network (MLP Regressor) Baseline...
Neural Net Fold 1 MRMSE Score: 1.18468


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


Neural Net Fold 2 MRMSE Score: 1.44487
Neural Net Fold 3 MRMSE Score: 1.25079

🏆 Overall Neural Network CV MRMSE: 1.29345


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


In [6]:
from sklearn.neural_network import MLPRegressor

mrmse_tuned_nn_scores = []

print("🚀 Training Fully Optimized Non-Linear Neural Network (MLP Regressor)...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_encoded_df)):
    X_train, X_val = X_encoded_df.iloc[train_idx], X_encoded_df.iloc[val_idx]
    y_train, y_val = Y_genes.iloc[train_idx], Y_genes.iloc[val_idx]
    
    # Increasing max_iter to allow full convergence and tweaking the learning rate
    tuned_nn = MLPRegressor(
        hidden_layer_sizes=(128, 64), 
        activation='relu', 
        solver='adam', 
        batch_size=32,
        learning_rate_init=0.005,
        max_iter=150, 
        early_stopping=True,
        n_iter_no_change=10, # Stops automatically if the validation score stops improving
        random_state=42
    )
    
    tuned_nn.fit(X_train, y_train)
    
    # Predict and evaluate
    tuned_preds = tuned_nn.predict(X_val)
    fold_score = mean_rowwise_rmse(y_val.values, tuned_preds)
    mrmse_tuned_nn_scores.append(fold_score)
    print(f"Optimized NN Fold {fold + 1} MRMSE Score: {fold_score:.5f}")

print(f"\n🏆 Overall Tuned Neural Network CV MRMSE: {np.mean(mrmse_tuned_nn_scores):.5f}")

🚀 Training Fully Optimized Non-Linear Neural Network (MLP Regressor)...
Optimized NN Fold 1 MRMSE Score: 1.31070
Optimized NN Fold 2 MRMSE Score: 1.43422
Optimized NN Fold 3 MRMSE Score: 1.28569

🏆 Overall Tuned Neural Network CV MRMSE: 1.34354


In [7]:
from sklearn.neural_network import MLPRegressor

mrmse_final_nn_scores = []

print("🚀 Training the Corrected & Fully Converged Neural Network...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_encoded_df)):
    X_train, X_val = X_encoded_df.iloc[train_idx], X_encoded_df.iloc[val_idx]
    y_train, y_val = Y_genes.iloc[train_idx], Y_genes.iloc[val_idx]
    
    # Keeping the stable default learning rate (0.001) and batch size,
    # but allowing more iterations (150) to fully converge without warnings.
    final_nn = MLPRegressor(
        hidden_layer_sizes=(128, 64), 
        activation='relu', 
        solver='adam', 
        learning_rate_init=0.001,  # Safe, stable learning rate
        max_iter=150,             # Enough time to reach the minimum
        early_stopping=True,       # Prevents overfitting automatically
        n_iter_no_change=10, 
        random_state=42
    )
    
    final_nn.fit(X_train, y_train)
    
    # Predict and evaluate using our rowwise MRMSE metric
    final_preds = final_nn.predict(X_val)
    fold_score = mean_rowwise_rmse(y_val.values, final_preds)
    mrmse_final_nn_scores.append(fold_score)
    print(f"Stable NN Fold {fold + 1} MRMSE Score: {fold_score:.5f}")

print(f"\n🏆 True Optimized Neural Network CV MRMSE: {np.mean(mrmse_final_nn_scores):.5f}")

🚀 Training the Corrected & Fully Converged Neural Network...
Stable NN Fold 1 MRMSE Score: 1.18468
Stable NN Fold 2 MRMSE Score: 1.41614
Stable NN Fold 3 MRMSE Score: 1.29421

🏆 True Optimized Neural Network CV MRMSE: 1.29834


In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.neural_network import MLPRegressor

print("--- 🏆 Final Phase: Full Training & Submission Generation ---")

# 1. Load the Test Map provided by the competition
id_map = pd.read_csv('/kaggle/input/competitions/open-problems-single-cell-perturbations/id_map.csv')

# 2. Re-fit the encoder on the entire training dataset for maximum coverage
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_full = encoder.fit_transform(de_train[['cell_type', 'sm_name']])
X_test_full = encoder.transform(id_map[['cell_type', 'sm_name']])

print(f"Full Train Shape: {X_train_full.shape}")
print(f"Full Test Shape: {X_test_full.shape}")

# 3. Train our stable Optimized Neural Network on 100% of the training data
final_model = MLPRegressor(
    hidden_layer_sizes=(128, 64), 
    activation='relu', 
    solver='adam', 
    learning_rate_init=0.001,
    max_iter=150, 
    early_stopping=True,
    random_state=42
)

print("\n🧠 Training the final production model...")
final_model.fit(X_train_full, Y_genes)

# 4. Predict the gene expressions for the test set (18,211 genes)
print("🔮 Generating predictions for the competition test set...")
test_preds = final_model.predict(X_test_full)

# 5. Format the submission exactly as required by Kaggle
submission = pd.DataFrame(test_preds, columns=Y_genes.columns)
submission.insert(0, 'id', id_map['id'])

# Save to CSV in the working directory
submission.to_csv('submission.csv', index=False)
print("\n💾 Success! 'submission.csv' generated and saved beautifully.")

--- 🏆 Final Phase: Full Training & Submission Generation ---
Full Train Shape: (614, 152)
Full Test Shape: (255, 152)

🧠 Training the final production model...
🔮 Generating predictions for the competition test set...

💾 Success! 'submission.csv' generated and saved beautifully.
